# visualize.py — YOLOv8 객체 탐지 실행 및 결과 시각화

## 이 파일은 무엇인가요?

이 파일은 **YOLOv8 모델을 사용하여 고양이 이미지에서 객체(물체)를 탐지하고, 그 결과를 시각화하는 도구**입니다.

메인 VTON 파이프라인과는 별도로 동작하는 **독립적인 유틸리티 스크립트**입니다. AutoML 대신 로컬에서 YOLOv8 모델을 직접 돌려보고 싶을 때 사용합니다.

### 이 스크립트가 하는 일
1. `pet-images/images/cat/cat2/` 폴더에 있는 고양이 사진들을 읽어옵니다.
2. YOLOv8 모델(`yolov8n.pt`)을 로드합니다.
3. 모든 이미지에 대해 객체 탐지를 수행합니다.
4. 탐지된 객체에 **바운딩 박스(네모 박스)가 그려진 이미지**를 `yolo_outputs/cat2/` 폴더에 저장합니다.
5. 탐지 결과를 **YOLO 포맷의 텍스트 파일(.txt)**로도 저장합니다.

### YOLOv8이란?
**YOLO (You Only Look Once)**는 실시간 객체 탐지 AI 모델입니다. 이미지 안에서 "이 물체가 무엇인지(클래스)" + "어디에 있는지(바운딩 박스)"를 동시에 알려줍니다.
- `yolov8n.pt`의 `n`은 **Nano(초소형)** 버전을 의미합니다. 가장 가볍고 빠르지만, 정확도는 상대적으로 낮습니다.
- 크기 순서: `n`(Nano) < `s`(Small) < `m`(Medium) < `l`(Large) < `x`(Extra Large)

### 사용된 기술
| 기술 | 역할 |
|---|---|
| **ultralytics** | YOLOv8 모델을 쉽게 로드하고 추론(탐지)을 수행할 수 있는 공식 파이썬 라이브러리입니다. |
| **pathlib.Path** | 파일/폴더 경로를 운영체제(Windows/Mac/Linux)에 관계없이 안전하게 다루는 파이썬 내장 도구입니다. |

## 1단계: 라이브러리 불러오기 및 경로 설정

- `Path`: 파이썬 내장 모듈로, 파일/폴더 경로를 객체로 다룹니다. 문자열 경로보다 안전하고 편리합니다.
  - `Path(__file__).resolve().parent.parent`: 이 파일의 위치를 기준으로 2단계 상위 폴더(프로젝트 루트)를 찾습니다.
  - `/` 연산자로 경로를 이어붙일 수 있습니다: `ROOT_DIR / "yolov8n.pt"` → `프로젝트루트/yolov8n.pt`
- `YOLO`: ultralytics 라이브러리에서 제공하는 YOLOv8 모델 클래스입니다.

### 경로 설명
| 변수 | 값 | 설명 |
|---|---|---|
| `ROOT_DIR` | 프로젝트 루트 폴더 | `Google_Pet To/` |
| `MODEL_PATH` | `ROOT_DIR/yolov8n.pt` | YOLOv8 Nano 모델 가중치 파일 경로 |
| `SOURCE_DIR` | `ROOT_DIR/pet-images/images/cat/cat2/` | 탐지할 고양이 이미지들이 있는 폴더 |
| `OUTPUT_PROJECT_DIR` | `ROOT_DIR/pet-images/yolo_outputs/` | 결과물이 저장될 상위 폴더 |
| `OUTPUT_RUN_NAME` | `"cat2"` | 이번 실행 결과가 저장될 하위 폴더 이름 |

In [ ]:
from pathlib import Path
from ultralytics import YOLO

ROOT_DIR = Path(__file__).resolve().parent.parent
MODEL_PATH = ROOT_DIR / "yolov8n.pt"
SOURCE_DIR = ROOT_DIR / "pet-images" / "images" / "cat" / "cat2"
OUTPUT_PROJECT_DIR = ROOT_DIR / "pet-images" / "yolo_outputs"
OUTPUT_RUN_NAME = "cat2"

## 2단계: 사전 검증 — 모델 파일과 이미지 폴더가 존재하는지 확인

실제 추론을 시작하기 전에, 필요한 파일과 폴더가 정상적으로 존재하는지 먼저 확인합니다.

1. **모델 파일 존재 확인**: `yolov8n.pt` 파일이 없으면 `FileNotFoundError`를 발생시킵니다.
2. **이미지 폴더 존재 확인**: `cat2/` 폴더가 없으면 에러를 발생시킵니다.
3. **이미지 파일 존재 확인**: 폴더 안에 `.jpg`, `.jpeg`, `.png`, `.webp` 확장자의 이미지 파일이 하나도 없으면 에러를 발생시킵니다.

> **왜 이렇게 검증하나요?** 모델이나 이미지가 없는 상태에서 YOLO를 돌리면 내부에서 알아보기 어려운 에러가 발생합니다. 미리 검증하면 원인을 명확하게 파악할 수 있습니다.

In [ ]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model file not found: {MODEL_PATH}")

if not SOURCE_DIR.exists():
    raise FileNotFoundError(f"Source directory not found: {SOURCE_DIR}")

image_files = [
    path for path in SOURCE_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
]
if not image_files:
    raise FileNotFoundError(f"No image files found in: {SOURCE_DIR}")

print(f"모델 파일: {MODEL_PATH}")
print(f"이미지 폴더: {SOURCE_DIR}")
print(f"발견된 이미지 수: {len(image_files)}")

## 3단계: YOLOv8 모델 로드 및 객체 탐지 실행

### 모델 로드
`YOLO(str(MODEL_PATH))`로 사전학습된 YOLOv8 Nano 모델을 로드합니다.

### `model.predict()` — 객체 탐지 실행
모든 이미지에 대해 한 번에 탐지를 수행합니다.

| 매개변수 | 값 | 설명 |
|---|---|---|
| `source` | `SOURCE_DIR` | 탐지할 이미지가 있는 폴더 경로 |
| `save=True` | True | 바운딩 박스가 그려진 이미지를 파일로 저장합니다 |
| `save_txt=True` | True | 탐지 결과를 YOLO 포맷의 텍스트 파일(.txt)로 저장합니다 |
| `save_conf=True` | True | 텍스트 파일에 **신뢰도(Confidence) 점수**도 함께 기록합니다 |
| `project` | `OUTPUT_PROJECT_DIR` | 결과물의 상위 폴더 경로 |
| `name` | `"cat2"` | 이번 실행 결과의 하위 폴더 이름 |
| `exist_ok=True` | True | 이미 같은 이름의 폴더가 있으면 덮어씁니다. (False면 `cat2_2`, `cat2_3` 처럼 새 폴더가 생김) |

### YOLO 텍스트 라벨(.txt) 형식
각 이미지마다 동일한 이름의 `.txt` 파일이 생성됩니다. 한 줄이 하나의 탐지 결과를 의미합니다:
```
클래스_ID  x_center  y_center  width  height  confidence
```
- 좌표와 크기는 0.0~1.0 사이의 **정규화된 값**입니다. (이미지 크기에 비례한 비율)
- 예: `15 0.512 0.634 0.324 0.456 0.89` → 클래스 15(고양이)가 이미지 중앙 부근에 89% 확신으로 탐지됨

In [ ]:
model = YOLO(str(MODEL_PATH))
results = model.predict(
    source=str(SOURCE_DIR),
    save=True,
    save_txt=True,
    save_conf=True,
    project=str(OUTPUT_PROJECT_DIR),
    name=OUTPUT_RUN_NAME,
    exist_ok=True,
)

## 4단계: 탐지 결과 요약 출력

모든 이미지에 대한 탐지가 끝난 후, 결과를 요약하여 출력합니다.

- **처리된 이미지 수**: 전체 이미지 파일 개수
- **탐지 성공 이미지 수**: 바운딩 박스가 1개 이상 잡힌 이미지 개수 (반려동물이 검출된 이미지)
- **결과 저장 경로**: 바운딩 박스가 그려진 이미지와 라벨 텍스트 파일이 저장된 폴더 경로

In [ ]:
detected_images = sum(1 for result in results if len(result.boxes) > 0)
output_dir = OUTPUT_PROJECT_DIR / OUTPUT_RUN_NAME
label_dir = output_dir / "labels"

print(f"처리된 이미지 수: {len(image_files)}")
print(f"탐지 성공 이미지 수: {detected_images}")
print(f"바운딩 박스 이미지 저장 경로: {output_dir}")
print(f"YOLO 라벨 파일 저장 경로: {label_dir}")